# LangChain Agent Examples Notebook

This notebook demonstrates all three LangChain agent implementations:
1. **Chat Agent with Memory** - Conversational AI with persistent memory
2. **Research Agent** - Web search and information synthesis
3. **Autonomous Workflow Agent** - Multi-step task execution with error handling

## Setup

Before running this notebook:
1. Install dependencies: `pip install -r requirements.txt`
2. Create `.env` file with your API keys
3. Load environment variables below

In [ ]:
# Setup and imports
import os
import sys
from pathlib import Path

# Add current directory to path
sys.path.insert(0, str(Path.cwd()))

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Verify API keys are loaded
openai_key = os.getenv('OPENAI_API_KEY')
print(f"OpenAI API Key loaded: {bool(openai_key)}")
print(f"Current working directory: {os.getcwd()}")

# 1. Chat Agent with Memory

A conversational agent that maintains both short-term and long-term memory for context-aware responses.

In [ ]:
from chat_agent_with_memory import ConversationalAgent

# Initialize the chat agent
chat_agent = ConversationalAgent(
    model="gpt-3.5-turbo",
    temperature=0.7,
    memory_window_size=5
)

print("Chat Agent initialized!")
print(f"Session ID: {chat_agent.session_id}")

## 1.1 Basic Conversation

In [ ]:
# Start a conversation
user_input = "Hi! My name is Alice and I'm interested in machine learning."
print(f"User: {user_input}")

response = chat_agent.chat(user_input, use_long_term_memory=True)
print(f"\nAgent: {response}")

In [ ]:
# Continue conversation
user_input = "Can you explain what deep learning is?"
print(f"\nUser: {user_input}")

response = chat_agent.chat(user_input, use_long_term_memory=True)
print(f"\nAgent: {response}")

In [ ]:
# Test memory recall
user_input = "What did you learn about me earlier?"
print(f"\nUser: {user_input}")

response = chat_agent.chat(user_input, use_long_term_memory=True)
print(f"\nAgent: {response}")

## 1.2 Session Management

In [ ]:
# Get session summary
summary = chat_agent.get_session_summary()
print("Session Summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

In [ ]:
# Export session
export_path = "./session_export.json"
chat_agent.export_session(export_path)
print(f"Session exported to {export_path}")

# 2. Research Agent

An intelligent research agent for web search, information synthesis, and source tracking.

In [ ]:
from search_research_agent import ResearchAgent

# Initialize research agent
research_agent = ResearchAgent(
    model="gpt-3.5-turbo",
    temperature=0.3
)

print("Research Agent initialized!")

## 2.1 Basic Research

In [ ]:
# Conduct research on a topic
question = "What are the key features of Python 3.11?"
print(f"Researching: {question}\n")

answer = research_agent.research(question, use_deep_research=False)

# Display results
print(f"Confidence Level: {answer.confidence_level}")
print(f"Number of Sources: {len(answer.sources)}")
print(f"\n{answer.to_markdown()}")

## 2.2 Deep Research

In [ ]:
# Conduct deeper research
question = "Latest developments in natural language processing"
print(f"Deep researching: {question}\n")

answer = research_agent.research(question, use_deep_research=True)

print(f"Confidence Level: {answer.confidence_level}")
print(f"Number of Sources: {len(answer.sources)}")
print(f"\n{answer.to_markdown()[:500]}...")  # Show first 500 chars

## 2.3 Research Export

In [ ]:
# Get research history
history = research_agent.get_research_history()
print("Research History:")
for item in history:
    print(f"  - {item['question']}")
    print(f"    Sources: {item['num_sources']}, Confidence: {item['confidence']}")

In [ ]:
# Export research findings
import os
os.makedirs("research_output", exist_ok=True)

research_agent.export_research("research_output/findings.md")
research_agent.export_research("research_output/findings.json")
print("Research exported to research_output/")

# 3. Autonomous Workflow Agent

An agent for executing complex multi-step workflows with error handling and task orchestration.

In [ ]:
from autonomous_workflow_agent import (
    AutonomousWorkflowAgent,
    Task,
    TaskPriority
)

# Initialize workflow agent
workflow_agent = AutonomousWorkflowAgent(
    model="gpt-3.5-turbo",
    temperature=0.3
)

print("Workflow Agent initialized!")

## 3.1 Simple Linear Workflow

In [ ]:
# Create simple linear workflow
tasks = [
    Task(
        task_id="step1",
        name="Data Collection",
        description="Gather input data",
        priority=TaskPriority.HIGH
    ),
    Task(
        task_id="step2",
        name="Data Processing",
        description="Clean and normalize data",
        dependencies=["step1"]
    ),
    Task(
        task_id="step3",
        name="Analysis",
        description="Analyze processed data",
        dependencies=["step2"]
    ),
]

print("Executing workflow...\n")
results = workflow_agent.execute_workflow(tasks)

In [ ]:
# Display results
print("Workflow Results:")
for task_id, result in results.items():
    print(f"\n{task_id}:")
    print(f"  Status: {result.status.value}")
    print(f"  Time: {result.execution_time:.2f}s")
    if result.output:
        print(f"  Output: {result.output}")

## 3.2 Complex Workflow with Dependencies

In [ ]:
# Create complex workflow with parallel branches
tasks = [
    Task(
        task_id="fetch",
        name="Fetch Data",
        description="Get data from API",
        priority=TaskPriority.HIGH
    ),
    Task(
        task_id="validate",
        name="Validate",
        description="Validate data integrity",
        dependencies=["fetch"]
    ),
    Task(
        task_id="analyze1",
        name="Analysis 1",
        description="Statistical analysis",
        dependencies=["validate"]
    ),
    Task(
        task_id="analyze2",
        name="Analysis 2",
        description="Pattern detection",
        dependencies=["validate"]
    ),
    Task(
        task_id="merge",
        name="Merge Results",
        description="Combine analysis results",
        dependencies=["analyze1", "analyze2"]
    ),
]

print("Executing complex workflow...\n")
results = workflow_agent.execute_workflow(tasks)
print("Complex workflow completed!")

## 3.3 Workflow Reporting

In [ ]:
# Get workflow report
report = workflow_agent.get_workflow_report()

print("Workflow Report:")
stats = report["statistics"]
print(f"  Total Tasks: {stats['total_tasks']}")
print(f"  Completed: {stats['completed']}")
print(f"  Failed: {stats['failed']}")
print(f"  Success Rate: {stats['success_rate']:.1%}")
print(f"  Total Time: {stats['total_time']:.2f}s")
print(f"  Average Time: {stats['average_time']:.2f}s")

In [ ]:
# Export workflow report
os.makedirs("workflow_reports", exist_ok=True)
workflow_agent.export_workflow_report("workflow_reports/report.json")
print("Workflow report exported to workflow_reports/report.json")

# Summary

This notebook demonstrated three powerful LangChain agent implementations:

1. **Chat Agent with Memory** - Maintains conversation context across sessions with persistent memory
2. **Research Agent** - Performs web research and synthesizes information from multiple sources
3. **Workflow Agent** - Executes complex multi-step workflows with error handling and dependency management

Each agent is production-ready with:
- Comprehensive error handling
- Type hints and documentation
- Configurable parameters
- Export and reporting capabilities

For more information, see the README.md file in the agents/examples directory.